In [1]:
import pandas as pd
import numpy as np

#Output file path
OUTPUT_FILE = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\MAC_DATASET_LLOD_FILTERED_V3.csv"

#read in original MAC dataset
df = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\MAC_NASA_DOE_3Bins_weather_originalbins.csv")

#Read in info files
df_llod_info = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\info_lists\LLOD_info_list_1.csv")
df_campaign_time_info = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\info_lists\campaign_time_info_list.csv")

model_columns = ['N1', 'N2', 'N3', 'N_total', 'V1', 'V2', 'V3', 'V_total', 'SAE', 'AAE', 'SSA_red', 'SSA_blue', 'SSA_green', 'b_scat_red', 'b_scat_blue', 'b_scat_green', 'b_abs_red', 'b_abs_blue', 'b_abs_green', 'MAC_bc']

In [ ]:
print("Original dataset shape:", df.shape)
print("\nLLOD Info:")
print(df_llod_info)
print("\nCampaign Time Info:")
print(df_campaign_time_info)

# Create a mapping from campaign to time averaging
campaign_time_map = dict(zip(df_campaign_time_info['campaign'], df_campaign_time_info['time_avg']))
print("\nCampaign to Time Averaging Mapping:")
for camp, time_avg in campaign_time_map.items():
    print(f"  {camp}: {time_avg}s")

# Create LLOD mapping based on time averaging
def get_llod_value(column_name, time_avg):
    """Get LLOD value for a column based on time averaging"""
    llod_row = df_llod_info[df_llod_info['column_name'] == column_name]
    if llod_row.empty:
        return None
    
    if time_avg == 1:
        return llod_row['llod_1s'].iloc[0]
    elif time_avg == 10:
        return llod_row['llod_10s'].iloc[0]
    elif time_avg == 20:
        return llod_row['llod_20s'].iloc[0]
    else:
        print(f"Warning: Unknown time averaging {time_avg}, using 1s LLOD")
        return llod_row['llod_1s'].iloc[0]

# Track filtering statistics
filtering_stats = {}
total_rows_before = len(df)

print(f"\nStarting LLOD filtering...")
print(f"Total rows before filtering: {total_rows_before}")
print("\n" + "="*60)

# Create a copy for filtering
df_filtered = df.copy()

# Get unique campaigns and their row counts
campaign_counts_before = df_filtered['Campaign'].value_counts().sort_index()
print("\nRows per campaign before filtering:")
for campaign, count in campaign_counts_before.items():
    print(f"  {campaign}: {count}")

# Process each campaign separately
for campaign in df_filtered['Campaign'].unique():
    campaign_mask = df_filtered['Campaign'] == campaign
    campaign_data = df_filtered[campaign_mask].copy()
    original_campaign_rows = len(campaign_data)
    
    # Get time averaging for this campaign
    time_avg = campaign_time_map.get(campaign)
    if time_avg is None:
        print(f"Warning: No time averaging found for campaign '{campaign}', skipping...")
        continue
    
    print(f"\nProcessing campaign: {campaign} (time_avg: {time_avg}s)")
    
    # Track rows to remove for this campaign
    rows_to_remove = pd.Series([False] * len(campaign_data), index=campaign_data.index)
    
    # Apply LLOD filtering for each column
    for _, llod_row in df_llod_info.iterrows():
        column_name = llod_row['column_name']
        
        if column_name not in campaign_data.columns:
            print(f"  Warning: Column '{column_name}' not found in dataset")
            continue
        
        llod_value = get_llod_value(column_name, time_avg)
        
        # First identify values below LLOD
        below_llod = (campaign_data[column_name] < llod_value)
        
        # Calculate 99th percentile ONLY on values >= LLOD (reliable measurements)
        valid_data = campaign_data[campaign_data[column_name] >= llod_value][column_name]
        
        if len(valid_data) > 0:
            percentile_99 = valid_data.quantile(0.99)
            # Remove values above 99th percentile (calculated from valid data only)
            above_99th = (campaign_data[column_name] > percentile_99)
        else:
            # If no valid data, don't apply upper limit filtering
            percentile_99 = float('inf')
            above_99th = pd.Series([False] * len(campaign_data), index=campaign_data.index)
            print(f"    Warning: No valid data >= LLOD for {column_name}, skipping upper limit filtering")
        
        # Combine both filters
        outside_range = below_llod | above_99th
        rows_to_remove = rows_to_remove | outside_range
        
        below_llod_count = below_llod.sum()
        above_99th_count = above_99th.sum()
        total_values = len(campaign_data[column_name].dropna())
        valid_count = len(valid_data)
        
        print(f"  {column_name}: LLOD={llod_value:.3f}, 99th percentile={percentile_99:.3f}")
        print(f"    Valid data (≥LLOD): {valid_count}/{total_values} ({100*valid_count/total_values:.1f}%)")
        print(f"    Below LLOD: {below_llod_count}/{total_values} ({100*below_llod_count/total_values:.1f}%)")
        print(f"    Above 99th: {above_99th_count}/{total_values} ({100*above_99th_count/total_values:.1f}%)")
    
    # Remove rows from the main dataframe
    rows_removed_count = rows_to_remove.sum()
    filtering_stats[campaign] = {
        'original_rows': original_campaign_rows,
        'rows_removed': rows_removed_count,
        'rows_remaining': original_campaign_rows - rows_removed_count,
        'percent_removed': 100 * rows_removed_count / original_campaign_rows
    }
    
    # Update the filtered dataframe
    df_filtered = df_filtered[~(campaign_mask & rows_to_remove)]
    
    print(f"  Removed {rows_removed_count}/{original_campaign_rows} rows "
          f"({100*rows_removed_count/original_campaign_rows:.1f}%)")

# Final statistics
total_rows_after = len(df_filtered)
total_rows_removed = total_rows_before - total_rows_after

print("\n" + "="*60)
print("FILTERING SUMMARY")
print("="*60)

print(f"\nOverall Statistics:")
print(f"  Total rows before: {total_rows_before:,}")
print(f"  Total rows after:  {total_rows_after:,}")
print(f"  Total rows removed: {total_rows_removed:,}")
print(f"  Percentage removed: {100*total_rows_removed/total_rows_before:.2f}%")

print(f"\nPer-Campaign Statistics:")
for campaign, stats in filtering_stats.items():
    print(f"  {campaign}:")
    print(f"    Original: {stats['original_rows']:,}")
    print(f"    Removed:  {stats['rows_removed']:,} ({stats['percent_removed']:.1f}%)")
    print(f"    Remaining: {stats['rows_remaining']:,}")

# Verify final campaign counts
print(f"\nRows per campaign after filtering:")
campaign_counts_after = df_filtered['Campaign'].value_counts().sort_index()
for campaign, count in campaign_counts_after.items():
    original_count = campaign_counts_before.get(campaign, 0)
    removed_count = original_count - count
    print(f"  {campaign}: {count} (removed {removed_count})")

print(f"\nColumns used for LLOD + 99th percentile filtering:")
for column in df_llod_info['column_name']:
    print(f"  - {column}")

print("\nFiltering removes:")
print("  - Values below LLOD (unreliable detections)")
print("  - Values above 99th percentile of reliable data (extreme outliers)")
print("  - Note: 99th percentile calculated only from measurements ≥ LLOD")
print("\nLLOD + 99th percentile filtering completed successfully!")

# Add spectral absorption order filtering
print("\n" + "="*60)
print("SPECTRAL ABSORPTION ORDER FILTERING")
print("="*60)

rows_before_spectral_filter = len(df_filtered)
print(f"Rows before spectral absorption filtering: {rows_before_spectral_filter:,}")

# Check if all required columns exist for absorption
required_abs_cols = ['b_abs_blue', 'b_abs_green', 'b_abs_red']
missing_abs_cols = [col for col in required_abs_cols if col not in df_filtered.columns]

if missing_abs_cols:
    print(f"Warning: Missing absorption columns for spectral filtering: {missing_abs_cols}")
    print("Skipping spectral absorption filtering...")
    rows_removed_spectral_abs = 0  # Track for final summary
else:
    # Apply spectral absorption order filter: b_abs_blue >= b_abs_green >= b_abs_red
    print("Applying filter: b_abs_blue >= b_abs_green >= b_abs_red")
    
    # Create the spectral order condition for absorption
    spectral_abs_condition = (
        (df_filtered['b_abs_blue'] >= df_filtered['b_abs_green']) & 
        (df_filtered['b_abs_green'] >= df_filtered['b_abs_red'])
    )
    
    # Count rows that meet/don't meet the condition
    rows_meeting_abs_condition = spectral_abs_condition.sum()
    rows_not_meeting_abs_condition = (~spectral_abs_condition).sum()
    
    print(f"  Rows meeting absorption spectral order: {rows_meeting_abs_condition:,} ({100*rows_meeting_abs_condition/rows_before_spectral_filter:.1f}%)")
    print(f"  Rows NOT meeting absorption condition: {rows_not_meeting_abs_condition:,} ({100*rows_not_meeting_abs_condition/rows_before_spectral_filter:.1f}%)")
    
    # Apply the filter
    df_filtered = df_filtered[spectral_abs_condition]
    
    rows_after_spectral_abs_filter = len(df_filtered)
    rows_removed_spectral_abs = rows_before_spectral_filter - rows_after_spectral_abs_filter
    
    print(f"\nSpectral absorption filtering results:")
    print(f"  Rows removed: {rows_removed_spectral_abs:,}")
    print(f"  Rows remaining: {rows_after_spectral_abs_filter:,}")
    print(f"  Percentage removed: {100*rows_removed_spectral_abs/rows_before_spectral_filter:.2f}%")
    
    print(f"\nSpectral absorption order filtering completed!")

# Add spectral scattering order filtering
print("\n" + "="*60)
print("SPECTRAL SCATTERING ORDER FILTERING")
print("="*60)

rows_before_spectral_scat_filter = len(df_filtered)
print(f"Rows before spectral scattering filtering: {rows_before_spectral_scat_filter:,}")

# Check if all required columns exist for scattering
required_scat_cols = ['b_scat_blue', 'b_scat_green', 'b_scat_red']
missing_scat_cols = [col for col in required_scat_cols if col not in df_filtered.columns]

if missing_scat_cols:
    print(f"Warning: Missing scattering columns for spectral filtering: {missing_scat_cols}")
    print("Skipping spectral scattering filtering...")
    rows_removed_spectral_scat = 0  # Track for final summary
else:
    # Apply spectral scattering order filter: b_scat_blue >= b_scat_green >= b_scat_red
    print("Applying filter: b_scat_blue >= b_scat_green >= b_scat_red")
    
    # Create the spectral order condition for scattering
    spectral_scat_condition = (
        (df_filtered['b_scat_blue'] >= df_filtered['b_scat_green']) & 
        (df_filtered['b_scat_green'] >= df_filtered['b_scat_red'])
    )
    
    # Count rows that meet/don't meet the condition
    rows_meeting_scat_condition = spectral_scat_condition.sum()
    rows_not_meeting_scat_condition = (~spectral_scat_condition).sum()
    
    print(f"  Rows meeting scattering spectral order: {rows_meeting_scat_condition:,} ({100*rows_meeting_scat_condition/rows_before_spectral_scat_filter:.1f}%)")
    print(f"  Rows NOT meeting scattering condition: {rows_not_meeting_scat_condition:,} ({100*rows_not_meeting_scat_condition/rows_before_spectral_scat_filter:.1f}%)")
    
    # Apply the filter
    df_filtered = df_filtered[spectral_scat_condition]
    
    rows_after_spectral_scat_filter = len(df_filtered)
    rows_removed_spectral_scat = rows_before_spectral_scat_filter - rows_after_spectral_scat_filter
    
    print(f"\nSpectral scattering filtering results:")
    print(f"  Rows removed: {rows_removed_spectral_scat:,}")
    print(f"  Rows remaining: {rows_after_spectral_scat_filter:,}")
    print(f"  Percentage removed: {100*rows_removed_spectral_scat/rows_before_spectral_scat_filter:.2f}%")
    
    print(f"\nSpectral scattering order filtering completed!")

# Combined spectral filtering impact
if 'rows_removed_spectral_abs' in locals() and 'rows_removed_spectral_scat' in locals():
    total_rows_removed_spectral = rows_removed_spectral_abs + rows_removed_spectral_scat
    
    print(f"\nCOMBINED SPECTRAL FILTERING IMPACT:")
    print(f"  Total rows before spectral filtering: {rows_before_spectral_filter:,}")
    print(f"  Rows removed by absorption filter: {rows_removed_spectral_abs:,}")
    print(f"  Rows removed by scattering filter: {rows_removed_spectral_scat:,}")
    print(f"  Total rows removed by spectral filtering: {total_rows_removed_spectral:,}")
    print(f"  Rows remaining after spectral filtering: {len(df_filtered):,}")
    print(f"  Total spectral filtering percentage: {100*total_rows_removed_spectral/rows_before_spectral_filter:.2f}%")
    
    # Check campaign-specific impact for combined spectral filtering
    print(f"\nCampaign-specific impact of combined spectral filtering:")
    campaign_counts_after_spectral = df_filtered['Campaign'].value_counts().sort_index()
    
    for campaign in campaign_counts_after.index:
        original_count = campaign_counts_after.get(campaign, 0)
        new_count = campaign_counts_after_spectral.get(campaign, 0)
        removed_count = original_count - new_count
        if removed_count > 0:
            print(f"  {campaign}: {new_count} (removed {removed_count}, {100*removed_count/original_count:.1f}%)")
        else:
            print(f"  {campaign}: {new_count} (no change)")

print("Proceeding to final empty value filtering...")

# Before saving, filter out rows with empty model_columns
print("\n" + "="*60)
print("FINAL EMPTY VALUE FILTERING")
print("="*60)

rows_before_empty_filter = len(df_filtered)
print(f"Rows before empty value filtering: {rows_before_empty_filter:,}")

# Create mask for valid data in model_columns
valid_mask = True
empty_stats = {}

for col in model_columns:
    if col in df_filtered.columns:
        # Check for empty, NaN, or invalid values
        col_mask = (~df_filtered[col].isna()) & (df_filtered[col] != '') & (df_filtered[col] != -2222) & (df_filtered[col] != -9999)
        
        # Count empty values
        empty_count = len(df_filtered) - col_mask.sum()
        empty_stats[col] = empty_count
        
        if empty_count > 0:
            print(f"  {col}: {empty_count} empty/invalid values")
        
        valid_mask = valid_mask & col_mask
    else:
        print(f"  Warning: Column '{col}' not found in dataset")

# Apply final filter
df_filtered = df_filtered[valid_mask]

rows_after_empty_filter = len(df_filtered)
rows_removed_empty = rows_before_empty_filter - rows_after_empty_filter

print(f"\nEmpty value filtering results:")
print(f"  Rows removed: {rows_removed_empty:,}")
print(f"  Rows remaining: {rows_after_empty_filter:,}")
print(f"  Percentage removed: {100*rows_removed_empty/rows_before_empty_filter:.2f}%")

# Complete filtering summary
print("\n" + "="*60)
print("COMPLETE FILTERING SUMMARY")
print("="*60)

# Calculate total rows removed across all steps
total_rows_original = total_rows_before
total_rows_after_llod = rows_before_spectral_filter
total_rows_after_spectral = len(df_filtered) + rows_removed_empty
total_rows_final = len(df_filtered)

# Calculate rows removed at each step
rows_removed_llod = total_rows_original - total_rows_after_llod
total_rows_removed_spectral = 0

# Track spectral filtering
if 'rows_removed_spectral_abs' in locals():
    total_rows_removed_spectral += rows_removed_spectral_abs
if 'rows_removed_spectral_scat' in locals():
    total_rows_removed_spectral += rows_removed_spectral_scat

print(f"FILTERING STEP BREAKDOWN:")
print(f"  1. Original dataset: {total_rows_original:,} rows")
print(f"  2. After LLOD + 99th percentile filtering: {total_rows_after_llod:,} rows")
print(f"     → Removed: {rows_removed_llod:,} rows ({100*rows_removed_llod/total_rows_original:.2f}%)")

if total_rows_removed_spectral > 0:
    print(f"  3. After spectral order filtering (absorption + scattering): {total_rows_after_spectral:,} rows")
    print(f"     → Removed: {total_rows_removed_spectral:,} rows ({100*total_rows_removed_spectral/total_rows_after_llod:.2f}%)")
    if 'rows_removed_spectral_abs' in locals() and 'rows_removed_spectral_scat' in locals():
        print(f"       • Absorption filter removed: {rows_removed_spectral_abs:,} rows")
        print(f"       • Scattering filter removed: {rows_removed_spectral_scat:,} rows")
    
print(f"  4. After empty value filtering: {total_rows_final:,} rows")
print(f"     → Removed: {rows_removed_empty:,} rows ({100*rows_removed_empty/total_rows_after_spectral:.2f}%)")

total_rows_removed_final = total_rows_original - total_rows_final
print(f"\nOVERALL SUMMARY:")
print(f"  Total rows removed: {total_rows_removed_final:,}")
print(f"  Total rows remaining: {total_rows_final:,}")
print(f"  Overall percentage removed: {100*total_rows_removed_final/total_rows_original:.2f}%")

print(f"\nFILTERING CRITERIA APPLIED:")
print(f"  ✓ LLOD filtering (values below detection limits)")
print(f"  ✓ 99th percentile outlier removal")
if 'rows_removed_spectral_abs' in locals() and rows_removed_spectral_abs > 0:
    print(f"  ✓ Spectral absorption order (b_abs_blue >= b_abs_green >= b_abs_red)")
if 'rows_removed_spectral_scat' in locals() and rows_removed_spectral_scat > 0:
    print(f"  ✓ Spectral scattering order (b_scat_blue >= b_scat_green >= b_scat_red)")
print(f"  ✓ Empty/invalid value removal")

# Final dataset info
print(f"\nFINAL DATASET CHARACTERISTICS:")
final_campaign_counts = df_filtered['Campaign'].value_counts().sort_index()
print(f"  Number of campaigns: {len(final_campaign_counts)}")
print(f"  Campaigns remaining: {', '.join(final_campaign_counts.index)}")
print(f"  Features: {len([col for col in model_columns if col in df_filtered.columns])}")

# Save filtered dataset
df_filtered.to_csv(OUTPUT_FILE, index=False)
print(f"\nFinal filtered dataset saved to: {OUTPUT_FILE}")
print("All filtering steps completed successfully!")

Original dataset shape: (1269014, 59)

LLOD Info:
    column_name  llod_1s  llod_10s  llod_20s
0    b_scat_red    3.165     1.635     1.290
1   b_scat_blue    5.490     2.790     2.220
2  b_scat_green    2.055     1.050     0.840
3     b_abs_red    4.860     0.237     0.096
4    b_abs_blue    4.860     0.237     0.096
5   b_abs_green    4.860     0.237     0.096

Campaign Time Info:
                 campaign  time_avg
0                 FIREXAQ         1
1                 SEAC4RS         1
2                 CAMP2Ex         1
3                 ASIA-AQ         1
4                     DC3         1
5           DISCOVERAQ-DC         1
6            NAAMES(2016)         1
7   DISCOVERAQ-California         1
8            NAAMES(2017)         1
9        DISCOVERAQ-Texas         1
10           NAAMES(2015)         1
11                ACE-ENA         1
12                  ISDAC         1
13               GOAMAZON         1
14                   BBOP        10
15                  ACMEV        10
16

In [ ]:
import pandas as pd
import numpy as np

# Read in the datasets
df = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\MAC_NASA_DOE_3Bins_weather_originalbins.csv")
df_llod_info = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\LLOD_info_list\LLOD_info_list.csv")

# Try different encodings for the campaign file
try:
    df_campaign_time_info = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\LLOD_info_list\campaign_time_info_list.csv", encoding='utf-8')
except UnicodeDecodeError:
    try:
        df_campaign_time_info = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\LLOD_info_list\campaign_time_info_list.csv", encoding='cp1252')
    except:
        df_campaign_time_info = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\LLOD_info_list\campaign_time_info_list.csv", encoding='latin1')

# Clean up campaign names
df_campaign_time_info['campaign'] = df_campaign_time_info['campaign'].str.replace('�', '', regex=False)

# Create campaign mapping
campaign_time_map = dict(zip(df_campaign_time_info['campaign'], df_campaign_time_info['time_avg']))

def get_llod_value(column_name, time_avg):
    llod_row = df_llod_info[df_llod_info['column_name'] == column_name]
    if llod_row.empty:
        return None
    
    if time_avg == 1:
        return llod_row['llod_1s'].iloc[0]
    elif time_avg == 10:
        return llod_row['llod_10s'].iloc[0]
    elif time_avg == 20:
        return llod_row['llod_20s'].iloc[0]
    else:
        return llod_row['llod_1s'].iloc[0]

print("DIAGNOSTIC ANALYSIS: Understanding Why 82% of Data is Below LLOD")
print("="*70)

# Check data ranges for each column
columns_to_check = ['b_scat_red', 'b_scat_blue', 'b_scat_green', 'b_abs_red', 'b_abs_blue', 'b_abs_green']

print("\n1. OVERALL DATA DISTRIBUTION ANALYSIS")
print("-" * 40)

for col in columns_to_check:
    if col in df.columns:
        data = df[col].dropna()
        print(f"\n{col}:")
        print(f"  Count: {len(data):,}")
        print(f"  Min: {data.min():.6f}")
        print(f"  Max: {data.max():.6f}")
        print(f"  Mean: {data.mean():.6f}")
        print(f"  Median: {data.median():.6f}")
        print(f"  25th percentile: {data.quantile(0.25):.6f}")
        print(f"  75th percentile: {data.quantile(0.75):.6f}")
        print(f"  95th percentile: {data.quantile(0.95):.6f}")
        print(f"  99th percentile: {data.quantile(0.99):.6f}")
        
        # Check for negative values
        negative_count = (data < 0).sum()
        print(f"  Negative values: {negative_count} ({100*negative_count/len(data):.1f}%)")
        
        # Check for zero values
        zero_count = (data == 0).sum()
        print(f"  Zero values: {zero_count} ({100*zero_count/len(data):.1f}%)")

print("\n\n2. LLOD VALUES vs DATA RANGES")
print("-" * 40)

print("\nLLOD Values:")
for _, row in df_llod_info.iterrows():
    col = row['column_name']
    print(f"{col}:")
    print(f"  1s LLOD: {row['llod_1s']:.3f}")
    print(f"  10s LLOD: {row['llod_10s']:.3f}")
    print(f"  20s LLOD: {row['llod_20s']:.3f}")

print("\n\n3. ALL CAMPAIGNS ANALYSIS")
print("-" * 40)

# Analyze ALL campaigns
all_campaigns = df['Campaign'].unique()

for campaign in sorted(all_campaigns):
    if campaign in df['Campaign'].values:
        campaign_data = df[df['Campaign'] == campaign]
        time_avg = campaign_time_map.get(campaign, 1)
        
        print(f"\n{campaign} (time_avg: {time_avg}s, {len(campaign_data)} rows):")
        
        for col in columns_to_check:
            if col in campaign_data.columns:
                data = campaign_data[col].dropna()
                llod = get_llod_value(col, time_avg)
                
                if len(data) > 0 and llod is not None:
                    below_llod = (data < llod).sum()
                    percent_below = 100 * below_llod / len(data)
                    
                    print(f"  {col}: LLOD={llod:.3f}")
                    print(f"    Data range: {data.min():.6f} to {data.max():.6f}")
                    print(f"    Median: {data.median():.6f}")
                    print(f"    Below LLOD: {below_llod}/{len(data)} ({percent_below:.1f}%)")

print("\n\n4. POTENTIAL ISSUES TO INVESTIGATE")
print("-" * 40)

print("\nCheck these potential issues:")
print("1. Are LLOD values too high for your measurement ranges?")
print("2. Are most measurements naturally very small (atmospheric background)?")
print("3. Are there unit mismatches (e.g., LLOD in different units than data)?")
print("4. Are the LLOD values from a different instrument setup?")
print("5. Should we be using different LLOD values for different environments?")

print("\n\n5. RECOMMENDATIONS")
print("-" * 40)

print("\nBased on the analysis above:")
print("- If most data is naturally below LLOD, the filtering might be correct")
print("- If LLOD values seem too high, consider adjusting them")
print("- If data ranges vary dramatically by campaign, consider campaign-specific LLODs")
print("- Consider if 82% removal is scientifically reasonable for your application")

print("\nDiagnostic analysis complete!")

DIAGNOSTIC ANALYSIS: Understanding Why 82% of Data is Below LLOD

1. OVERALL DATA DISTRIBUTION ANALYSIS
----------------------------------------

b_scat_red:
  Count: 1,269,014
  Min: -9086.859280
  Max: 5419.580030
  Mean: 40.072410
  Median: 10.148657
  25th percentile: 1.761574
  75th percentile: 31.566509
  95th percentile: 130.348713
  99th percentile: 411.848412
  Negative values: 98857 (7.8%)
  Zero values: 116 (0.0%)

b_scat_blue:
  Count: 1,269,014
  Min: -9083.821496
  Max: 12876.458593
  Mean: 85.207809
  Median: 19.883559
  25th percentile: 3.133353
  75th percentile: 64.667543
  95th percentile: 237.744310
  99th percentile: 916.280638
  Negative values: 98862 (7.8%)
  Zero values: 97 (0.0%)

b_scat_green:
  Count: 1,269,014
  Min: -9084.930086
  Max: 9105.758288
  Mean: 64.417488
  Median: 15.328082
  25th percentile: 2.378916
  75th percentile: 49.565814
  95th percentile: 189.576882
  99th percentile: 684.520962
  Negative values: 98861 (7.8%)
  Zero values: 101 (0.0%)
